# 27 — Clean and validate global LIANA labels

**Purpose:** Reproduce the historical Notebook 24b post-processing semantics on the
canonical Notebook 26 outputs, without rerunning LIANA.

**Historical counterpart:** `24b_fix_LIANA_global_main_labels_and_axes_HVG8000_r0_6_v2_fixed_source_clean.ipynb`

Notebook 25 now fixes the historical `Malignant_non_malignant_*` substring-label bug
upstream, so Notebook 26 already contains the canonical cell-type labels. Therefore
Notebook 27 must **not** reintroduce the historical `NonMalignant_*` naming style.
It validates the canonical labels, rebuilds the historical clean-label summaries with
explicit deterministic group mappings, and checks that the regenerated summaries agree
with Notebook 26.

Run from top to bottom in a fresh kernel.


## Historical semantics and current canonical interpretation

Historical Notebook 24b was post-processing only. It did not rerun LIANA; it repaired
misleading broad-axis classifications caused by malformed non-malignant labels.

In the canonical pipeline:
- LIANA scores/ranks come exclusively from Notebook 26.
- `source` and `target` labels remain the canonical Notebook 25/26 labels.
- Broad groups are assigned from an explicit 9-cell-type map.
- No file auto-detection or fallback table substitution is allowed.
- Historical malformed labels and historical `NonMalignant_*` aliases are treated as
  invalid upstream residues and cause a hard failure.
- Figures are diagnostic continuity outputs and should not be staged during Main Analysis.


In [1]:
from pathlib import Path
from datetime import datetime
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


In [2]:
# Fixed repository paths and canonical constants
def _find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
FIG_DIR = PROJECT_DIR / "figures" / "LIANA_HVG8000_r0_6"
FIG_DIR.mkdir(parents=True, exist_ok=True)

OLD_PREFIX = "LIANA_global_main_HVG8000_r0_6"
OUT_PREFIX = "LIANA_global_main_HVG8000_r0_6_clean_labels"

RANK_SIG_PATH = METADATA_DIR / f"{OLD_PREFIX}_rank_threshold_or_top_interactions.tsv"
TOP_GLOBAL_PATH = METADATA_DIR / f"{OLD_PREFIX}_top_global_interactions.tsv"
TOP_PER_PAIR_PATH = METADATA_DIR / f"{OLD_PREFIX}_top_source_target_interactions.tsv"
FULL_RESULTS_PATH = METADATA_DIR / f"{OLD_PREFIX}_all_results.tsv.gz"
COUNTS_PATH = METADATA_DIR / f"{OLD_PREFIX}_celltype_counts_used.tsv"
RUN_SUMMARY_PATH = METADATA_DIR / f"{OLD_PREFIX}_run_summary.tsv"
NB26_SOURCE_TARGET_PATH = METADATA_DIR / f"{OLD_PREFIX}_source_target_summary.tsv"
NB26_AXIS_PATH = METADATA_DIR / f"{OLD_PREFIX}_axis_summary.tsv"
NB26_BIO_PATH = METADATA_DIR / f"{OLD_PREFIX}_biological_axis_summary.tsv"
NB26_LR_PATH = METADATA_DIR / f"{OLD_PREFIX}_ligand_receptor_summary.tsv"
NB26_ANNOTATED_PATH = METADATA_DIR / f"{OLD_PREFIX}_prioritized_interactions_annotated.tsv"

EXPECTED_MAIN_COUNTS = {
    "Malignant_OPC_NPC_like_malignant": 61646,
    "Malignant_cycling_candidate": 38634,
    "Malignant_hypoxia_stress_malignant": 32896,
    "Malignant_AC_like_reactive_malignant": 22525,
    "Oligodendrocyte_myelinating": 14809,
    "Malignant_NPC_neuronal_like_malignant": 6183,
    "Mural_Pericyte_Fibroblast": 3566,
    "Endothelial": 2640,
    "B_Plasma": 1167,
}

MALIGNANT_CELLTYPES = {
    "Malignant_OPC_NPC_like_malignant",
    "Malignant_cycling_candidate",
    "Malignant_hypoxia_stress_malignant",
    "Malignant_AC_like_reactive_malignant",
    "Malignant_NPC_neuronal_like_malignant",
}
MICROENVIRONMENT_CELLTYPES = set(EXPECTED_MAIN_COUNTS) - MALIGNANT_CELLTYPES
EXPECTED_CELLTYPES = set(EXPECTED_MAIN_COUNTS)

LINEAGE_MAP = {
    **{x: "malignant_state" for x in MALIGNANT_CELLTYPES},
    "Endothelial": "endothelial",
    "Mural_Pericyte_Fibroblast": "mural_pericyte_fibroblast",
    "Oligodendrocyte_myelinating": "oligodendrocyte_reference",
    "B_Plasma": "B_plasma",
}

EXPECTED_NOTEBOOK26_CHECKPOINTS = {
    "n_cells_used": 184066,
    "n_genes_used": 15176,
    "n_liana_celltypes": 9,
    "n_liana_results": 23797,
    "n_prioritized_or_rank_threshold_interactions": 1384,
    "n_unique_lr_pairs_prioritized": 264,
    "n_source_target_pairs_prioritized": 81,
    "n_communication_axes": 4,
}

print("PROJECT_DIR:", PROJECT_DIR)
print("RANK_SIG_PATH:", RANK_SIG_PATH)


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
RANK_SIG_PATH: G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_rank_threshold_or_top_interactions.tsv


In [3]:
# Deterministic helper functions
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print("[WRITE]", path, df.shape)
    return path

def savefig(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("[FIG]", path)
    return path

def canonical_broad_group(label):
    label = str(label)
    if label in MALIGNANT_CELLTYPES:
        return "malignant_or_cycling"
    if label in MICROENVIRONMENT_CELLTYPES:
        return "non_malignant_reference_like"
    raise ValueError(f"Unexpected LIANA cell type: {label!r}")

def canonical_lineage(label):
    label = str(label)
    if label not in LINEAGE_MAP:
        raise ValueError(f"Unexpected LIANA cell type for lineage mapping: {label!r}")
    return LINEAGE_MAP[label]

def biological_axis_from_groups(source_group, target_group):
    mapping = {
        ("malignant_or_cycling", "non_malignant_reference_like"):
            "malignant_to_microenvironment",
        ("non_malignant_reference_like", "malignant_or_cycling"):
            "microenvironment_to_malignant",
        ("malignant_or_cycling", "malignant_or_cycling"):
            "malignant_state_to_malignant_state",
        ("non_malignant_reference_like", "non_malignant_reference_like"):
            "microenvironment_to_microenvironment",
    }
    key = (str(source_group), str(target_group))
    if key not in mapping:
        raise ValueError(f"Unexpected broad-group pair: {key}")
    return mapping[key]

def require_columns(df, required, table_name):
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{table_name}: missing required columns {sorted(missing)}")

def assert_no_legacy_labels(values, context):
    values = pd.Series(values, dtype="string").dropna().astype(str)
    malformed = values.str.contains("Malignant_non_malignant", regex=False)
    old_clean = values.str.startswith("NonMalignant_")
    if malformed.any() or old_clean.any():
        bad = sorted(values[malformed | old_clean].unique().tolist())
        raise AssertionError(
            f"{context}: historical label residue detected: {bad[:20]}"
        )

def frames_match(left, right, keys, numeric_cols, string_cols=(), atol=1e-12):
    l = left.copy()
    r = right.copy()
    l = l.sort_values(keys).reset_index(drop=True)
    r = r.sort_values(keys).reset_index(drop=True)
    if len(l) != len(r):
        return False, f"row_count {len(l)} != {len(r)}"
    if l[keys].astype(str).equals(r[keys].astype(str)) is False:
        return False, "key mismatch"
    for c in string_cols:
        if not l[c].astype(str).equals(r[c].astype(str)):
            return False, f"string column mismatch: {c}"
    for c in numeric_cols:
        if not np.allclose(
            pd.to_numeric(l[c], errors="coerce"),
            pd.to_numeric(r[c], errors="coerce"),
            rtol=0,
            atol=atol,
            equal_nan=True,
        ):
            return False, f"numeric column mismatch: {c}"
    return True, "match"


In [4]:
# Read exact Notebook 26 inputs; no fallbacks or optional substitutions.
required_inputs = {
    "rank_threshold_or_top_interactions": RANK_SIG_PATH,
    "top_global_interactions": TOP_GLOBAL_PATH,
    "top_source_target_interactions": TOP_PER_PAIR_PATH,
    "all_results": FULL_RESULTS_PATH,
    "celltype_counts_used": COUNTS_PATH,
    "run_summary": RUN_SUMMARY_PATH,
    "source_target_summary": NB26_SOURCE_TARGET_PATH,
    "axis_summary": NB26_AXIS_PATH,
    "biological_axis_summary": NB26_BIO_PATH,
    "ligand_receptor_summary": NB26_LR_PATH,
    "prioritized_interactions_annotated": NB26_ANNOTATED_PATH,
}

readiness = pd.DataFrame([
    {
        "item": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else np.nan,
    }
    for label, path in required_inputs.items()
])
write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")
display(readiness)

if not readiness["exists"].all():
    missing = readiness.loc[~readiness["exists"], ["item", "path"]]
    raise FileNotFoundError(
        "Missing required Notebook 26 canonical outputs:\n" + missing.to_string(index=False)
    )

rank_sig = pd.read_csv(RANK_SIG_PATH, sep="\t")
top_global = pd.read_csv(TOP_GLOBAL_PATH, sep="\t")
top_per_pair = pd.read_csv(TOP_PER_PAIR_PATH, sep="\t")
counts = pd.read_csv(COUNTS_PATH, sep="\t")
run_summary = pd.read_csv(RUN_SUMMARY_PATH, sep="\t")
nb26_source_target = pd.read_csv(NB26_SOURCE_TARGET_PATH, sep="\t")
nb26_axis = pd.read_csv(NB26_AXIS_PATH, sep="\t")
nb26_bio = pd.read_csv(NB26_BIO_PATH, sep="\t")
nb26_lr = pd.read_csv(NB26_LR_PATH, sep="\t")
nb26_annotated = pd.read_csv(NB26_ANNOTATED_PATH, sep="\t")

required_interaction_cols = {
    "source", "target", "ligand_complex", "receptor_complex",
    "lr_pair", "primary_rank", "primary_rank_col",
}
for name, df in [
    ("rank_sig", rank_sig),
    ("top_global", top_global),
    ("top_per_pair", top_per_pair),
]:
    require_columns(df, required_interaction_cols, name)
    assert_no_legacy_labels(df["source"], f"{name}.source")
    assert_no_legacy_labels(df["target"], f"{name}.target")

observed_rank_celltypes = set(rank_sig["source"].astype(str)).union(
    set(rank_sig["target"].astype(str))
)
if observed_rank_celltypes != EXPECTED_CELLTYPES:
    raise AssertionError(
        "Notebook 26 prioritized interactions do not contain the exact 9 canonical cell types.\n"
        f"Missing: {sorted(EXPECTED_CELLTYPES - observed_rank_celltypes)}\n"
        f"Unexpected: {sorted(observed_rank_celltypes - EXPECTED_CELLTYPES)}"
    )

if len(rank_sig) != EXPECTED_NOTEBOOK26_CHECKPOINTS[
    "n_prioritized_or_rank_threshold_interactions"
]:
    raise AssertionError(f"Unexpected prioritized row count: {len(rank_sig)}")

summary_map = dict(zip(run_summary["item"].astype(str), run_summary["value"]))
summary_audit_rows = []
for item, expected in EXPECTED_NOTEBOOK26_CHECKPOINTS.items():
    if item not in summary_map:
        raise ValueError(f"Notebook 26 run_summary is missing {item!r}")
    observed = pd.to_numeric(pd.Series([summary_map[item]]), errors="coerce").iloc[0]
    match = bool(pd.notna(observed) and float(observed) == float(expected))
    summary_audit_rows.append({
        "checkpoint": item,
        "expected": expected,
        "observed": observed,
        "match": match,
    })
summary_audit = pd.DataFrame(summary_audit_rows)
if not summary_audit["match"].all():
    raise AssertionError(
        "Notebook 26 run-summary checkpoints failed:\n"
        + summary_audit.to_string(index=False)
    )

display(summary_audit)
print("[rank_sig]", rank_sig.shape)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_readiness.tsv (11, 4)


,item,path,exists,size_mb
0,rank_threshold_or_top_interactions,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.449
1,top_global_interactions,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.307
2,top_source_target_interactions,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.740
3,all_results,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,1.456
4,celltype_counts_used,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
5,run_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
6,source_target_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.016
7,axis_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
8,biological_axis_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.003
9,ligand_receptor_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.019


,checkpoint,expected,observed,match
0,n_cells_used,184066,184066,True
1,n_genes_used,15176,15176,True
2,n_liana_celltypes,9,9,True
3,n_liana_results,23797,23797,True
4,n_prioritized_or_rank_threshold_interactions,1384,1384,True
5,n_unique_lr_pairs_prioritized,264,264,True
6,n_source_target_pairs_prioritized,81,81,True
7,n_communication_axes,4,4,True


[rank_sig] (1384, 18)


In [5]:
# Validate canonical labels and rebuild historical clean-label annotations.
require_columns(counts, {"liana_celltype", "observed_n_cells"}, "celltype_counts_used")

count_map = dict(
    zip(counts["liana_celltype"].astype(str), pd.to_numeric(counts["observed_n_cells"]))
)
if set(count_map) != EXPECTED_CELLTYPES:
    raise AssertionError(
        "Cell-type count table does not contain the exact 9 canonical cell types."
    )

label_audit_rows = []
for label, expected_n in EXPECTED_MAIN_COUNTS.items():
    observed_n = int(count_map[label])
    if observed_n != expected_n:
        raise AssertionError(
            f"Cell-count mismatch for {label}: expected {expected_n}, observed {observed_n}"
        )
    label_audit_rows.append({
        "liana_celltype": label,
        "expected_n_cells": expected_n,
        "observed_n_cells": observed_n,
        "count_match": True,
        "canonical_broad_group": canonical_broad_group(label),
        "canonical_lineage": canonical_lineage(label),
        "historical_label_repair_applied": False,
    })

label_audit = pd.DataFrame(label_audit_rows)
write_tsv_replace(
    label_audit,
    METADATA_DIR / f"{OUT_PREFIX}_canonical_label_audit.tsv",
)
display(label_audit)

def apply_canonical_annotations(df, table_name):
    df = df.copy()

    # Traceability columns preserve the exact Notebook 26 labels.
    df["source_original"] = df["source"].astype(str)
    df["target_original"] = df["target"].astype(str)
    df["source_clean"] = df["source_original"]
    df["target_clean"] = df["target_original"]

    assert_no_legacy_labels(df["source_clean"], f"{table_name}.source_clean")
    assert_no_legacy_labels(df["target_clean"], f"{table_name}.target_clean")

    observed = set(df["source_clean"]).union(set(df["target_clean"]))
    unexpected = observed - EXPECTED_CELLTYPES
    if unexpected:
        raise AssertionError(f"{table_name}: unexpected canonical labels {sorted(unexpected)}")

    source_group = df["source_clean"].map(canonical_broad_group)
    target_group = df["target_clean"].map(canonical_broad_group)
    communication_axis = source_group + "_to_" + target_group

    # If Notebook 26 already supplied these columns, require exact agreement.
    if "source_broad_group" in df.columns:
        if not df["source_broad_group"].astype(str).equals(source_group.astype(str)):
            raise AssertionError(f"{table_name}: source_broad_group disagrees with canonical mapping")
    if "target_broad_group" in df.columns:
        if not df["target_broad_group"].astype(str).equals(target_group.astype(str)):
            raise AssertionError(f"{table_name}: target_broad_group disagrees with canonical mapping")
    if "communication_axis" in df.columns:
        if not df["communication_axis"].astype(str).equals(communication_axis.astype(str)):
            raise AssertionError(f"{table_name}: communication_axis disagrees with canonical mapping")

    df["source_broad_group"] = source_group
    df["target_broad_group"] = target_group
    df["communication_axis"] = communication_axis
    df["biological_axis"] = [
        biological_axis_from_groups(s, t)
        for s, t in zip(df["source_broad_group"], df["target_broad_group"])
    ]
    df["source_lineage_simplified"] = df["source_clean"].map(canonical_lineage)
    df["target_lineage_simplified"] = df["target_clean"].map(canonical_lineage)
    return df

rank_sig_clean = apply_canonical_annotations(rank_sig, "rank_sig")
top_global_clean = apply_canonical_annotations(top_global, "top_global")
top_per_pair_clean = apply_canonical_annotations(top_per_pair, "top_per_pair")

write_tsv_replace(
    rank_sig_clean,
    METADATA_DIR / f"{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv",
)
write_tsv_replace(
    top_global_clean,
    METADATA_DIR / f"{OUT_PREFIX}_top_global_interactions.tsv",
)
write_tsv_replace(
    top_per_pair_clean,
    METADATA_DIR / f"{OUT_PREFIX}_top_source_target_interactions.tsv",
)

counts_clean = counts.copy()
counts_clean["liana_celltype_original"] = counts_clean["liana_celltype"].astype(str)
counts_clean["broad_group"] = counts_clean["liana_celltype"].map(canonical_broad_group)
counts_clean["lineage_simplified"] = counts_clean["liana_celltype"].map(canonical_lineage)
write_tsv_replace(
    counts_clean,
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv",
)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_canonical_label_audit.tsv (9, 7)


,liana_celltype,expected_n_cells,observed_n_cells,count_match,canonical_broad_group,canonical_lineage,historical_label_repair_applied
0,Malignant_OPC_NPC_like_malignant,61646,61646,True,malignant_or_cycling,malignant_state,False
1,Malignant_cycling_candidate,38634,38634,True,malignant_or_cycling,malignant_state,False
2,Malignant_hypoxia_stress_malignant,32896,32896,True,malignant_or_cycling,malignant_state,False
3,Malignant_AC_like_reactive_malignant,22525,22525,True,malignant_or_cycling,malignant_state,False
4,Oligodendrocyte_myelinating,14809,14809,True,non_malignant_reference_like,oligodendrocyte_reference,False
5,Malignant_NPC_neuronal_like_malignant,6183,6183,True,malignant_or_cycling,malignant_state,False
6,Mural_Pericyte_Fibroblast,3566,3566,True,non_malignant_reference_like,mural_pericyte_fibroblast,False
7,Endothelial,2640,2640,True,non_malignant_reference_like,endothelial,False
8,B_Plasma,1167,1167,True,non_malignant_reference_like,B_plasma,False


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_rank_threshold_or_top_interactions.tsv (1384, 25)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_top_global_interactions.tsv (1000, 24)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_top_source_target_interactions.tsv (2430, 24)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_celltype_counts_used.tsv (9, 7)


WindowsPath('G:/Repository/glioma-cnv-single-cell-atlas/metadata/LIANA_global_main_HVG8000_r0_6_clean_labels_celltype_counts_used.tsv')

In [6]:
# Rebuild source-target and communication-axis summaries.
source_target_summary = (
    rank_sig_clean
    .groupby(
        ["source", "target", "source_broad_group", "target_broad_group",
         "communication_axis", "biological_axis"],
        observed=True,
    )
    .agg(
        n_prioritized_interactions=("lr_pair", "nunique"),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(
        ["n_prioritized_interactions", "median_primary_rank"],
        ascending=[False, True],
    )
)

axis_summary = (
    rank_sig_clean
    .groupby("communication_axis", observed=True)
    .agg(
        n_prioritized_interactions=("lr_pair", "nunique"),
        n_source_target_pairs=(
            "source",
            lambda x: len(set(zip(
                rank_sig_clean.loc[x.index, "source"],
                rank_sig_clean.loc[x.index, "target"],
            ))),
        ),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(
        ["n_prioritized_interactions", "median_primary_rank"],
        ascending=[False, True],
    )
)

expected_axes = {
    "malignant_or_cycling_to_malignant_or_cycling",
    "malignant_or_cycling_to_non_malignant_reference_like",
    "non_malignant_reference_like_to_malignant_or_cycling",
    "non_malignant_reference_like_to_non_malignant_reference_like",
}
observed_axes = set(axis_summary["communication_axis"].astype(str))
if observed_axes != expected_axes:
    raise AssertionError(
        f"Expected four canonical communication axes, observed {sorted(observed_axes)}"
    )

write_tsv_replace(
    source_target_summary,
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
)
write_tsv_replace(
    axis_summary,
    METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv",
)

# Compare regenerated Notebook 27 summaries to canonical Notebook 26 summaries.
st_compare = source_target_summary.drop(columns=["biological_axis"]).copy()
ok_st, msg_st = frames_match(
    st_compare,
    nb26_source_target,
    keys=["source", "target", "source_broad_group", "target_broad_group", "communication_axis"],
    numeric_cols=["n_prioritized_interactions", "median_primary_rank", "min_primary_rank"],
)
ok_axis, msg_axis = frames_match(
    axis_summary,
    nb26_axis,
    keys=["communication_axis"],
    numeric_cols=[
        "n_prioritized_interactions", "n_source_target_pairs",
        "median_primary_rank", "min_primary_rank",
    ],
)

if not ok_st:
    raise AssertionError(f"Notebook 27 source-target summary drifted from Notebook 26: {msg_st}")
if not ok_axis:
    raise AssertionError(f"Notebook 27 axis summary drifted from Notebook 26: {msg_axis}")

display(axis_summary)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_source_target_summary.tsv (81, 9)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_axis_summary.tsv (4, 5)


,communication_axis,n_prioritized_interactions,n_source_target_pairs,median_primary_rank,min_primary_rank
3,non_malignant_reference_like_to_non_malignant_...,168,16,0.015616,1.430284e-07
2,non_malignant_reference_like_to_malignant_or_c...,98,20,0.017271,2.542656e-07
1,malignant_or_cycling_to_non_malignant_referenc...,88,20,0.017740,1.073610e-05
0,malignant_or_cycling_to_malignant_or_cycling,60,25,0.012463,1.589294e-08


In [7]:
# Rebuild biological-axis and ligand-receptor summaries.
bio_summary = (
    rank_sig_clean
    .groupby(
        ["biological_axis", "source_lineage_simplified", "target_lineage_simplified"],
        observed=True,
    )
    .agg(
        n_interactions=("lr_pair", "nunique"),
        n_source_target_pairs=(
            "source",
            lambda x: len(set(zip(
                rank_sig_clean.loc[x.index, "source"],
                rank_sig_clean.loc[x.index, "target"],
            ))),
        ),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_interactions", "median_primary_rank"], ascending=[False, True])
)

lr_summary = (
    rank_sig_clean
    .groupby(["ligand_complex", "receptor_complex", "lr_pair"], observed=True)
    .agg(
        n_source_target_pairs=(
            "source",
            lambda x: len(set(zip(
                rank_sig_clean.loc[x.index, "source"],
                rank_sig_clean.loc[x.index, "target"],
            ))),
        ),
        n_sources=("source", "nunique"),
        n_targets=("target", "nunique"),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_source_target_pairs", "median_primary_rank"], ascending=[False, True])
)

write_tsv_replace(
    bio_summary,
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
)
write_tsv_replace(
    lr_summary,
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
)

ok_bio, msg_bio = frames_match(
    bio_summary,
    nb26_bio,
    keys=["biological_axis", "source_lineage_simplified", "target_lineage_simplified"],
    numeric_cols=[
        "n_interactions", "n_source_target_pairs",
        "median_primary_rank", "min_primary_rank",
    ],
)
ok_lr, msg_lr = frames_match(
    lr_summary,
    nb26_lr,
    keys=["ligand_complex", "receptor_complex", "lr_pair"],
    numeric_cols=[
        "n_source_target_pairs", "n_sources", "n_targets",
        "median_primary_rank", "min_primary_rank",
    ],
)

# Row-wise biological-axis cross-check against Notebook 26 annotated prioritized output.
require_columns(
    nb26_annotated,
    {"source", "target", "ligand_complex", "receptor_complex", "biological_axis"},
    "Notebook26 prioritized_interactions_annotated",
)
row_check_cols = ["source", "target", "ligand_complex", "receptor_complex"]
left = rank_sig_clean[row_check_cols + ["biological_axis"]].copy()
right = nb26_annotated[row_check_cols + ["biological_axis"]].copy()
left = left.sort_values(row_check_cols).reset_index(drop=True)
right = right.sort_values(row_check_cols).reset_index(drop=True)
ok_row_axis = (
    len(left) == len(right)
    and left[row_check_cols].astype(str).equals(right[row_check_cols].astype(str))
    and left["biological_axis"].astype(str).equals(right["biological_axis"].astype(str))
)

consistency_audit = pd.DataFrame([
    {"check": "source_target_summary_matches_notebook26", "match": ok_st, "detail": msg_st},
    {"check": "axis_summary_matches_notebook26", "match": ok_axis, "detail": msg_axis},
    {"check": "biological_axis_summary_matches_notebook26", "match": ok_bio, "detail": msg_bio},
    {"check": "ligand_receptor_summary_matches_notebook26", "match": ok_lr, "detail": msg_lr},
    {"check": "rowwise_biological_axis_matches_notebook26", "match": ok_row_axis,
     "detail": "match" if ok_row_axis else "row-wise axis mismatch"},
])
write_tsv_replace(
    consistency_audit,
    METADATA_DIR / f"{OUT_PREFIX}_notebook26_consistency_audit.tsv",
)

if not consistency_audit["match"].all():
    raise AssertionError(
        "Notebook 27 regenerated summaries do not match Notebook 26:\n"
        + consistency_audit.to_string(index=False)
    )

display(consistency_audit)
display(bio_summary.head(30))


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_biological_axis_summary.tsv (25, 7)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_ligand_receptor_summary.tsv (264, 8)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_notebook26_consistency_audit.tsv (5, 3)


,check,match,detail
0,source_target_summary_matches_notebook26,True,match
1,axis_summary_matches_notebook26,True,match
2,biological_axis_summary_matches_notebook26,True,match
3,ligand_receptor_summary_matches_notebook26,True,match
4,rowwise_biological_axis_matches_notebook26,True,match


,biological_axis,source_lineage_simplified,target_lineage_simplified,n_interactions,n_source_target_pairs,median_primary_rank,min_primary_rank
7,microenvironment_to_malignant,mural_pericyte_fibroblast,malignant_state,69,5,0.013717,2.542656e-07
19,microenvironment_to_microenvironment,mural_pericyte_fibroblast,mural_pericyte_fibroblast,63,1,0.013817,1.430284e-07
0,malignant_state_to_malignant_state,malignant_state,malignant_state,60,25,0.012463,1.589294e-08
18,microenvironment_to_microenvironment,mural_pericyte_fibroblast,endothelial,54,1,0.017783,5.146859e-06
3,malignant_to_microenvironment,malignant_state,mural_pericyte_fibroblast,43,5,0.017740,2.668609e-05
2,malignant_to_microenvironment,malignant_state,endothelial,38,5,0.019379,8.685173e-05
6,microenvironment_to_malignant,endothelial,malignant_state,36,5,0.025268,3.513761e-04
14,microenvironment_to_microenvironment,endothelial,endothelial,30,1,0.022652,1.157751e-05
20,microenvironment_to_microenvironment,mural_pericyte_fibroblast,oligodendrocyte_reference,29,1,0.021864,7.786231e-07
17,microenvironment_to_microenvironment,mural_pericyte_fibroblast,B_plasma,27,1,0.007378,4.291310e-05


In [8]:
# Focused historical post-processing outputs.
mal_to_micro = rank_sig_clean.loc[
    rank_sig_clean["biological_axis"].eq("malignant_to_microenvironment")
].copy()
micro_to_mal = rank_sig_clean.loc[
    rank_sig_clean["biological_axis"].eq("microenvironment_to_malignant")
].copy()
mal_to_mal = rank_sig_clean.loc[
    rank_sig_clean["biological_axis"].eq("malignant_state_to_malignant_state")
].copy()
micro_to_micro = rank_sig_clean.loc[
    rank_sig_clean["biological_axis"].eq("microenvironment_to_microenvironment")
].copy()

if len(mal_to_micro) + len(micro_to_mal) + len(mal_to_mal) + len(micro_to_micro) != len(rank_sig_clean):
    raise AssertionError("Biological-axis partition does not cover all prioritized interactions.")

write_tsv_replace(
    mal_to_micro,
    METADATA_DIR / f"{OUT_PREFIX}_malignant_to_microenvironment_interactions.tsv",
)
write_tsv_replace(
    micro_to_mal,
    METADATA_DIR / f"{OUT_PREFIX}_microenvironment_to_malignant_interactions.tsv",
)
write_tsv_replace(
    mal_to_mal,
    METADATA_DIR / f"{OUT_PREFIX}_malignant_state_to_malignant_state_interactions.tsv",
)

focused_summary = pd.DataFrame([
    {
        "axis": name,
        "n_rows": len(df),
        "n_unique_lr_pairs": df["lr_pair"].nunique(),
        "n_source_target_pairs": len(set(zip(df["source"], df["target"]))),
        "median_primary_rank": df["primary_rank"].median() if len(df) else np.nan,
        "min_primary_rank": df["primary_rank"].min() if len(df) else np.nan,
    }
    for name, df in [
        ("malignant_to_microenvironment", mal_to_micro),
        ("microenvironment_to_malignant", micro_to_mal),
        ("malignant_state_to_malignant_state", mal_to_mal),
    ]
])
write_tsv_replace(
    focused_summary,
    METADATA_DIR / f"{OUT_PREFIX}_focused_axis_summary.tsv",
)
display(focused_summary)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_malignant_to_microenvironment_interactions.tsv (309, 25)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_microenvironment_to_malignant_interactions.tsv (344, 25)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_malignant_state_to_malignant_state_interactions.tsv (391, 25)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_focused_axis_summary.tsv (3, 6)


,axis,n_rows,n_unique_lr_pairs,n_source_target_pairs,median_primary_rank,min_primary_rank
0,malignant_to_microenvironment,309,88,20,0.017740,1.073610e-05
1,microenvironment_to_malignant,344,98,20,0.017271,2.542656e-07
2,malignant_state_to_malignant_state,391,60,25,0.012463,1.589294e-08


In [9]:
# Diagnostic figures for historical continuity; do not stage during Main Analysis.
figure_records = []

pivot_counts = (
    source_target_summary
    .pivot(index="source", columns="target", values="n_prioritized_interactions")
    .fillna(0)
)
row_order = pivot_counts.sum(axis=1).sort_values(ascending=False).index
col_order = pivot_counts.sum(axis=0).sort_values(ascending=False).index
pivot_counts = pivot_counts.loc[row_order, col_order]

plt.figure(figsize=(max(10, 0.55 * pivot_counts.shape[1]), max(8, 0.45 * pivot_counts.shape[0])))
plt.imshow(pivot_counts.values, aspect="auto")
plt.colorbar(label="Unique prioritized LR pairs")
plt.xticks(range(pivot_counts.shape[1]), pivot_counts.columns, rotation=90, fontsize=8)
plt.yticks(range(pivot_counts.shape[0]), pivot_counts.index, fontsize=8)
plt.title("LIANA global main canonical labels: prioritized LR pairs")
plt.xlabel("Target cell type")
plt.ylabel("Source cell type")
fig_path = savefig(
    FIG_DIR / f"{OUT_PREFIX}_source_target_prioritized_count_heatmap.png"
)
figure_records.append({
    "figure": str(fig_path),
    "description": "Diagnostic canonical-label heatmap of unique prioritized LR pairs by source-target pair.",
})

axis_plot = axis_summary.sort_values("n_prioritized_interactions", ascending=True)
plt.figure(figsize=(9, max(4, 0.45 * len(axis_plot))))
plt.barh(axis_plot["communication_axis"], axis_plot["n_prioritized_interactions"])
plt.xlabel("Unique prioritized LR pairs")
plt.ylabel("Communication axis")
plt.title("LIANA global main canonical communication axes")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_axis_summary_barplot.png")
figure_records.append({
    "figure": str(fig_path),
    "description": "Diagnostic canonical-label barplot of unique prioritized LR pairs across communication axes.",
})

bio_top = bio_summary.head(20).sort_values("n_interactions", ascending=True).copy()
bio_top["label"] = (
    bio_top["source_lineage_simplified"] + " → " + bio_top["target_lineage_simplified"]
)
plt.figure(figsize=(9, max(5, 0.35 * len(bio_top))))
plt.barh(bio_top["label"], bio_top["n_interactions"])
plt.xlabel("Unique prioritized LR pairs")
plt.ylabel("Lineage axis")
plt.title("LIANA global main canonical labels: top lineage axes")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_top_lineage_axis_barplot.png")
figure_records.append({
    "figure": str(fig_path),
    "description": "Diagnostic canonical-label barplot of top lineage-level LIANA axes.",
})

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(
    figure_manifest,
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
)
display(figure_manifest)


[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_HVG8000_r0_6\LIANA_global_main_HVG8000_r0_6_clean_labels_source_target_prioritized_count_heatmap.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_HVG8000_r0_6\LIANA_global_main_HVG8000_r0_6_clean_labels_axis_summary_barplot.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_HVG8000_r0_6\LIANA_global_main_HVG8000_r0_6_clean_labels_top_lineage_axis_barplot.png
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_figure_manifest.tsv (3, 2)


,figure,description
0,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic canonical-label heatmap of unique p...
1,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic canonical-label barplot of unique p...
2,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic canonical-label barplot of top line...


In [10]:
# Run summary and exact output manifest.
summary = pd.DataFrame([
    {"item": "postprocessing_only", "value": True},
    {"item": "liana_rerun", "value": False},
    {"item": "original_prefix", "value": OLD_PREFIX},
    {"item": "clean_prefix", "value": OUT_PREFIX},
    {"item": "canonical_labels_already_fixed_upstream", "value": True},
    {"item": "historical_label_repair_applied", "value": False},
    {"item": "n_prioritized_rows", "value": rank_sig_clean.shape[0]},
    {"item": "n_unique_lr_pairs", "value": rank_sig_clean["lr_pair"].nunique()},
    {"item": "n_source_target_pairs", "value": len(set(zip(rank_sig_clean["source"], rank_sig_clean["target"])))},
    {"item": "n_clean_celltypes", "value": len(set(rank_sig_clean["source"]).union(set(rank_sig_clean["target"])))},
    {"item": "n_communication_axes", "value": len(observed_axes)},
    {"item": "notebook26_consistency_all_match", "value": bool(consistency_audit["match"].all())},
    {"item": "completed_at", "value": datetime.now().isoformat(timespec="seconds")},
])
write_tsv_replace(
    summary,
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
)

manifest_paths = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_canonical_label_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_notebook26_consistency_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_top_global_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_top_source_target_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_malignant_to_microenvironment_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_microenvironment_to_malignant_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_malignant_state_to_malignant_state_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_focused_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
]
manifest = pd.DataFrame([
    {
        "path": str(p),
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024 / 1024, 3) if p.exists() else np.nan,
    }
    for p in manifest_paths
])
write_tsv_replace(
    manifest,
    METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv",
)

if not manifest["exists"].all():
    raise AssertionError("One or more expected Notebook 27 outputs are missing.")

display(summary)
display(manifest)
print("[DONE] Notebook 27 canonical label post-processing completed.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_run_summary.tsv (13, 2)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_clean_labels_output_manifest.tsv (17, 3)


,item,value
0,postprocessing_only,True
1,liana_rerun,False
2,original_prefix,LIANA_global_main_HVG8000_r0_6
3,clean_prefix,LIANA_global_main_HVG8000_r0_6_clean_labels
4,canonical_labels_already_fixed_upstream,True
5,historical_label_repair_applied,False
6,n_prioritized_rows,1384
7,n_unique_lr_pairs,264
8,n_source_target_pairs,81
9,n_clean_celltypes,9


,path,exists,size_mb
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.002
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
3,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
4,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.679
5,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.474
6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,1.140
7,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.018
8,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
9,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.003


[DONE] Notebook 27 canonical label post-processing completed.


## Interpretation notes

- This notebook is **post-processing only** and never reruns LIANA.
- The historical malformed `Malignant_non_malignant_*` labels are no longer expected,
  because Notebook 25 fixes them upstream.
- Canonical labels are preserved exactly; historical `NonMalignant_*` aliases are not
  reintroduced.
- The regenerated source-target, communication-axis, biological-axis and LR summaries
  must agree with Notebook 26. Any disagreement causes a hard failure.
- Use the `_clean_labels` outputs as the Notebook 27 canonical post-processing products.
- Diagnostic figures are generated for continuity but remain parked until the figure phase.
